In [1]:
!pip install wfdb resampy ishneholterlib

In [ ]:
if not os.path.exists("/content/CMED-Mini-Project"):
    !git clone https://github.com/Anonymous0002396/CMED-Mini-Project.git /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

Cloning into 'CMED-Mini-Project'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 116 (delta 5), reused 114 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 9.51 MiB | 30.24 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [4]:
import sys
from pathlib import Path

project_root = Path("/content/CMED-Mini-Project")
repo_root = project_root / "SSSD-ECG"

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "ptb_xl"))
sys.path.insert(0, str(repo_root / "src" / "sssd"))

print(project_root)
print(repo_root.exists())

/content/CMED-Mini-Project
True


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

from pathlib import Path

In [5]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

if not os.path.exists('/content/synthetic_ptbxl'): # Replace with the actual folder name inside the zip
    !cp "/content/drive/MyDrive/Dataset.zip" /content/synthetic_ptbxl.zip
    !unzip -oq /content/synthetic_ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("Dataset already exists, skipping extraction.")

Extraction complete.


In [21]:
if not os.path.exists('/content/ptbxl'): # Replace with the actual folder name inside the zip
    !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip
    !unzip -oq /content/ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("Dataset already exists, skipping extraction.")

Extraction complete.


### Set up paths

In [6]:
target_fs = 100

# raw real PTB-XL folder you just extracted
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")

# processed output folder that load_dataset(...) will read from later
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

print("raw exists:", data_folder_ptb_xl.exists())
print("processed exists:", target_folder_ptb_xl.exists())

raw exists: True
processed exists: True


### Real PTB-XL preprocessing

In [7]:
df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)

print(df_ptb_xl.shape)
print(type(lbl_itos_ptb_xl))
print(target_folder_ptb_xl)

  0%|          | 0/21799 [00:00<?, ?it/s]

(21799, 44)
<class 'dict'>
/content/processed_ptb_xl_fs100


In [8]:
reformat_as_memmap(
    df_ptb_xl,
    target_folder_ptb_xl / "memmap.npy",
    data_folder=target_folder_ptb_xl,
    delete_npys=True
)

  0%|          | 0/21799 [00:00<?, ?it/s]

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,label_diag_numeric,label_form_numeric,label_rhythm_numeric,label_diag_subclass_numeric,label_diag_superclass_numeric,data,data_mean,data_std,data_length,data_original
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,[37],[7],[7],[14],[3],0,"[0.0015079858, 0.0007229996, -0.0017089996, 0....","[0.1090222, 0.083223335, 0.11311742, 0.2142296...",1000,00001_lr.npy
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,[37],[],[6],[14],[3],1,"[-0.0021810099, -0.00016502624, -0.00040496505...","[0.13342775, 0.24974498, 0.23233838, 0.4500496...",1000,00002_lr.npy
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,[37],[],[7],[14],[3],2,"[-0.0027700001, 0.005145004, -0.00023700125, 0...","[0.12119952, 0.14438936, 0.11632309, 0.1476423...",1000,00003_lr.npy
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,[37],[],[7],[14],[3],3,"[-0.003263996, 0.003501991, -0.00078800163, -0...","[0.13780256, 0.43816054, 0.18206717, 0.4213087...",1000,00004_lr.npy
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,[37],[],[7],[14],[3],4,"[-0.003769, -0.003054002, 0.0016899869, 2.3997...","[0.08270639, 0.23678097, 0.14218302, 0.3419054...",1000,00005_lr.npy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,[36],"[8, 13, 18]",[8],[20],[4],21794,"[-0.010336004, -0.0058109984, -0.0007689993, -...","[0.21377766, 0.22204638, 0.13152157, 0.2788991...",1000,21833_lr.npy
21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,[37],[0],[7],[14],[3],21795,"[-0.0024299999, -0.0038730002, -0.0013400039, ...","[0.09910399, 0.08042687, 0.0962622, 0.19413967...",1000,21834_lr.npy
21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,[24],[],[7],[6],[4],21796,"[0.0005970012, -0.0027460016, -0.0006459985, -...","[0.096423626, 0.10434124, 0.13238312, 0.167256...",1000,21835_lr.npy


In [10]:
df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)

print(type(df_mapped), df_mapped.shape)
print(type(lbl_itos_dict))
print(lbl_itos_dict.keys())

<class 'pandas.DataFrame'> (21799, 45)
<class 'dict'>
dict_keys(['label_all', 'label_diag', 'label_form', 'label_rhythm', 'label_diag_subclass', 'label_diag_superclass'])


In [14]:
label_key = "label_all" 
label_names = np.array(lbl_itos_dict[label_key])

print(label_key, len(label_names))
print(label_names[:30])

label_all 71
['1AVB' '2AVB' '3AVB' 'ABQRS' 'AFIB' 'AFLT' 'ALMI' 'AMI' 'ANEUR' 'ASMI'
 'BIGU' 'CLBBB' 'CRBBB' 'DIG' 'EL' 'HVOLT' 'ILBBB' 'ILMI' 'IMI' 'INJAL'
 'INJAS' 'INJIL' 'INJIN' 'INJLA' 'INVT' 'IPLMI' 'IPMI' 'IRBBB' 'ISCAL'
 'ISCAN']


### Load synthetic dataset

In [16]:
# released synthetic PTB-XL dataset root after unzip
synthetic_root = Path("/content/Dataset")

syn_train_data_path = synthetic_root / "data" / "ptbxl_train_data.npy"
syn_val_data_path   = synthetic_root / "data" / "ptbxl_validation_data.npy"
syn_test_data_path  = synthetic_root / "data" / "ptbxl_test_data.npy"

syn_train_lbl_path = synthetic_root / "labels" / "ptbxl_train_labels.npy"
syn_val_lbl_path   = synthetic_root / "labels" / "ptbxl_validation_labels.npy"
syn_test_lbl_path  = synthetic_root / "labels" / "ptbxl_test_labels.npy"

for p in [
    syn_train_data_path, syn_val_data_path, syn_test_data_path,
    syn_train_lbl_path, syn_val_lbl_path, syn_test_lbl_path
]:
    print(p, "->", p.exists())

/content/Dataset/data/ptbxl_train_data.npy -> True
/content/Dataset/data/ptbxl_validation_data.npy -> True
/content/Dataset/data/ptbxl_test_data.npy -> True
/content/Dataset/labels/ptbxl_train_labels.npy -> True
/content/Dataset/labels/ptbxl_validation_labels.npy -> True
/content/Dataset/labels/ptbxl_test_labels.npy -> True


In [17]:
syn_train_data = np.load(syn_train_data_path)
syn_train_labels = np.load(syn_train_lbl_path)

print("syn_train_data:", syn_train_data.shape, syn_train_data.dtype)
print("syn_train_labels:", syn_train_labels.shape, syn_train_labels.dtype)

syn_train_data: (17441, 12, 1000) float32
syn_train_labels: (17441, 71) float32


### Label mapping + MI target creation

In [24]:
mi_statement_names = [
    "IMI", "ASMI", "ILMI", "AMI", "ALMI",
    "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
    "INJIN", "PMI", "INJLA", "INJIL"
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]
mi_label_names = [str(label_names[i]) for i in mi_label_indices]

print("MI labels:", mi_label_names)

MI labels: ['IMI', 'ASMI', 'ILMI', 'AMI', 'ALMI', 'INJAS', 'LMI', 'INJAL', 'IPLMI', 'IPMI', 'INJIN', 'PMI', 'INJLA', 'INJIL']


In [25]:
def multilabel_to_binary_mi(y_multihot, mi_indices):
    return (y_multihot[:, mi_indices].sum(axis=1) > 0).astype(np.float32)

syn_train_mi = multilabel_to_binary_mi(syn_train_labels, mi_label_indices)


In [27]:

# optional, if you already loaded these
syn_val_data = np.load(syn_val_data_path)
syn_val_labels = np.load(syn_val_lbl_path)
syn_test_data = np.load(syn_test_data_path)
syn_test_labels = np.load(syn_test_lbl_path)

syn_val_mi = multilabel_to_binary_mi(syn_val_labels, mi_label_indices)
syn_test_mi = multilabel_to_binary_mi(syn_test_labels, mi_label_indices)

print("Synthetic train MI prevalence:", syn_train_mi.mean(), syn_train_mi.sum(), len(syn_train_mi))
print("Synthetic val   MI prevalence:", syn_val_mi.mean(), syn_val_mi.sum(), len(syn_val_mi))
print("Synthetic test  MI prevalence:", syn_test_mi.mean(), syn_test_mi.sum(), len(syn_test_mi))

Synthetic train MI prevalence: 0.25164843 4389.0 17441
Synthetic val   MI prevalence: 0.24806201 544.0 2193
Synthetic test  MI prevalence: 0.25102133 553.0 2203


### Create multihot labels for the real PTB-XL metadata in the same 71-label space

In [28]:
def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out

df_real = df_mapped.copy()
df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
    lambda x: multihot_encode(x, len(label_names))
)
Y_real = np.stack(df_real["label_multihot"].values, axis=0)
df_real["label_mi"] = multilabel_to_binary_mi(Y_real, mi_label_indices)

In [32]:
df_real["label_multihot"][3]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)

### Recreate the standard real train/val/test splits

In [33]:
max_fold_id = df_real.strat_fold.max()

df_train_real = df_real[df_real.strat_fold < max_fold_id - 1].copy()
df_val_real   = df_real[df_real.strat_fold == max_fold_id - 1].copy()
df_test_real  = df_real[df_real.strat_fold == max_fold_id].copy()

print("Real split sizes:", len(df_train_real), len(df_val_real), len(df_test_real))

Real split sizes: 17418 2183 2198


### Create binary MI targets for the real PTB-XL splits

In [ ]:
def row_multihot_to_binary_mi(row_vec, mi_indices):
    return float(row_vec[mi_indices].sum() > 0)

for split_df in [df_train_real, df_val_real, df_test_real]:
    split_df["label_mi"] = split_df["label_multihot"].apply(
        lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
    )

print("Real train MI prevalence:", df_train_real["label_mi"].mean(), df_train_real["label_mi"].sum(), len(df_train_real))
print("Real val   MI prevalence:", df_val_real["label_mi"].mean(), df_val_real["label_mi"].sum(), len(df_val_real))
print("Real test  MI prevalence:", df_test_real["label_mi"].mean(), df_test_real["label_mi"].sum(), len(df_test_real))

Real train MI prevalence: 0.2514065908829946 4379.0 17418
Real val   MI prevalence: 0.24736601007787448 540.0 2183
Real test  MI prevalence: 0.2502274795268426 550.0 2198


### Add female/male masks for later evaluation

In [35]:
print(df_test_real["sex"].value_counts(dropna=False))

female_test_mask = df_test_real["sex"].astype(str).str.lower().eq("female").to_numpy()
male_test_mask   = df_test_real["sex"].astype(str).str.lower().eq("male").to_numpy()

print("Female test count:", female_test_mask.sum())
print("Male test count:", male_test_mask.sum())

sex
0    1132
1    1066
Name: count, dtype: int64
Female test count: 0
Male test count: 0


### Memmap alignment block

In [43]:
import pickle

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
    df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()
df_memmap_meta["label_multihot"] = df_real["label_multihot"].values
df_memmap_meta["label_mi"] = df_real["label_mi"].values
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values
df_memmap_meta["sex"] = df_real["sex"].values

max_fold_id = df_memmap_meta["strat_fold"].max()

df_train_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] < max_fold_id - 1].copy()
df_val_real_mem   = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem  = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

print(len(df_train_real_mem), len(df_val_real_mem), len(df_test_real_mem))

KeyError: 'label_mi'

### Select same 8 leads from real dataset, that is used in the synthetic dataset

In [36]:
lead_idx_8 = np.array([0, 2, 3, 4, 5, 6, 7, 11], dtype=np.int64)
print("Using lead indices:", lead_idx_8)

Using lead indices: [ 0  2  3  4  5  6  7 11]


### Build a simple Dataset classes

In [40]:
class RealPTBXL8LeadMIDataset(Dataset):
    def __init__(self, split_df, lead_idx_8):
        self.df = split_df.reset_index(drop=True).copy()
        self.lead_idx_8 = np.asarray(lead_idx_8, dtype=np.int64)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # row["data"] is expected to be shape [1000, 12]
        signal_12 = np.asarray(row["data"], dtype=np.float32)

        # convert to [8, 1000]
        signal_8 = signal_12[:, self.lead_idx_8].T

        label = np.float32(row["label_mi"])

        return {
            "signal": torch.tensor(signal_8, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.float32),
        }

In [41]:
class SyntheticPTBXL8LeadMIDataset(Dataset):
    def __init__(self, signals, labels_binary):
        self.signals = np.asarray(signals, dtype=np.float32)
        self.labels = np.asarray(labels_binary, dtype=np.float32)

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        signal = self.signals[idx]   # expected [8, 1000]
        label = self.labels[idx]

        return {
            "signal": torch.tensor(signal, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.float32),
        }

### Instantiate datasets

In [42]:
lead_idx_8 = np.array([0, 2, 3, 4, 5, 6, 7, 11], dtype=np.int64)

train_real_ds = RealPTBXL8LeadMIDataset(df_train_real, lead_idx_8)
val_real_ds   = RealPTBXL8LeadMIDataset(df_val_real, lead_idx_8)
test_real_ds  = RealPTBXL8LeadMIDataset(df_test_real, lead_idx_8)

batch = train_real_ds[0]
print(batch["signal"].shape, batch["label"])

IndexError: too many indices for array: array is 0-dimensional, but 2 were indexed